Import Libraries

In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Load Dataset

In [97]:
df = pd.read_csv("E:\\AIML-Projects\\gamegpt\\data\\steam.csv")

In [98]:
df.head()

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,1273,267,258,184,5000000-10000000,3.99
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,FPS;Action;Sci-fi,0,5250,288,624,415,5000000-10000000,3.99


In [99]:
# Shape of Dataset
print(f"Shape = {df.shape}")

Shape = (27075, 18)


In [100]:
print(df.columns.tolist())

['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners', 'price']


In [101]:
# Checking NULL Values
print(f"NULL Values:\n{df.isnull().sum()}")

NULL Values:
appid                0
name                 0
release_date         0
english              0
developer            1
publisher           14
platforms            0
required_age         0
categories           0
genres               0
steamspy_tags        0
achievements         0
positive_ratings     0
negative_ratings     0
average_playtime     0
median_playtime      0
owners               0
price                0
dtype: int64


In [102]:
# Checking Positive Ratings Column
print(df['positive_ratings'].describe())

count    2.707500e+04
mean     1.000559e+03
std      1.898872e+04
min      0.000000e+00
25%      6.000000e+00
50%      2.400000e+01
75%      1.260000e+02
max      2.644404e+06
Name: positive_ratings, dtype: float64


In [103]:
# Checking Owners Column 
print(df['owners'].value_counts().head(10))

owners
0-20000              18596
20000-50000           3059
50000-100000          1695
100000-200000         1386
200000-500000         1272
500000-1000000         513
1000000-2000000        288
2000000-5000000        193
5000000-10000000        46
10000000-20000000       21
Name: count, dtype: int64


Convert Owners Column from '0-10' format to '5' format

In [104]:
def convert_owners(owners_str):
    try:
        lower, higher = owners_str.split('-')
        return (int(lower) + int(higher)) / 2 
    except:
        return None
    
df['owners_midpoint'] = df['owners'].apply(convert_owners)
print(df['owners_midpoint'].describe())

count    2.707500e+04
mean     1.340905e+05
std      1.328089e+06
min      1.000000e+04
25%      1.000000e+04
50%      1.000000e+04
75%      3.500000e+04
max      1.500000e+08
Name: owners_midpoint, dtype: float64


In [105]:
def calculate_review_score(row):
    total = row['positive_ratings'] + row['negative_ratings']
    if total==0:
        return None
    else:
        return (row['positive_ratings']/total)*100
    
df['review_score'] = df.apply(calculate_review_score, axis=1)
print(f"review Score: \n{df['review_score'].head()}\n")
print(df['review_score'].describe())

review Score: 
0    97.388815
1    83.978740
2    89.564761
3    82.662338
4    94.799567
Name: review_score, dtype: float64

count    27075.000000
mean        71.447792
std         23.359421
min          0.000000
25%         58.333333
50%         76.033058
75%         89.390531
max        100.000000
Name: review_score, dtype: float64


In [106]:
games = ['Grand Theft Auto', 'Forza Horizon', 'Minecraft', 'Detroit', 'F1 2']
for game in games:
    result = df[df['name'].str.contains(game, na=False)][['name', 'positive_ratings', 'negative_ratings', 'owners_midpoint', 'review_score']]
    print(result)
    print("---")

                                              name  positive_ratings  \
288                           Grand Theft Auto III              4718   
289                    Grand Theft Auto: Vice City              9817   
290                  Grand Theft Auto: San Andreas             26877   
294                               Grand Theft Auto               199   
295                             Grand Theft Auto 2               184   
297                            Grand Theft Auto IV             35240   
298   Grand Theft Auto: Episodes from Liberty City              6401   
2478                            Grand Theft Auto V            329061   

      negative_ratings  owners_midpoint  review_score  
288                750        1500000.0     86.283833  
289                819        3500000.0     92.299737  
290               3243        3500000.0     89.233068  
294                 30         750000.0     86.899563  
295                 17         750000.0     91.542289  
297            

In [107]:
# Convert release_dte to datetime
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# Drop 15 rows with missing developer/publisher
df = df.dropna(subset=['developer', 'publisher'])

# Confirm Final Shape
print(f"Final clean shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")

Final clean shape: (27061, 20)
Missing values:
appid               0
name                0
release_date        0
english             0
developer           0
publisher           0
platforms           0
required_age        0
categories          0
genres              0
steamspy_tags       0
achievements        0
positive_ratings    0
negative_ratings    0
average_playtime    0
median_playtime     0
owners              0
price               0
owners_midpoint     0
review_score        0
dtype: int64


In [108]:
df.to_csv('../data/steam_clean.csv', index = False)
print('Saved!')

Saved!
